Make sure the right schema is used

In [0]:
USE CATALOG sac;
USE SCHEMA customer_service;

In [0]:
select zone, count(*) from customer group by zone

zone,count(1)
Thüringen,5978
Schleswig-Holstein,11185
Hamburg,3345
Sachsen,12273
Hessen,46
Berlin,4793
Nordrhein-Westfalen,52045
Brandenburg,10143
null,45
Bayern,11


In [0]:
select address, c.plz, bundesland, city from customer_bronze b join customer c on c.customer_id = b.customer_id join plz_to_state p on c.plz = p.plz where c.zone = 'Sachsen-Anhalt'

# Gold Tables
average customer

In [0]:
CREATE OR REPLACE VIEW average_customer AS
SELECT
    zone,
    COUNT(*) AS amount_customers,
    ROUND(AVG(monthly_bill), 2) AS avg_monthly_bill,
    ROUND(AVG(speed_tier_mbps), 0) AS avg_speed_tier,
    ROUND(AVG(data_usage_gb_last_month), 2) AS avg_data_usage
FROM
    customer
GROUP BY
    zone;

tickets per customer

In [0]:
CREATE OR REPLACE VIEW customer_connection_ticket_count AS
SELECT
    c.customer_id,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats,
    ch.churned as churned
FROM
    customer c
        JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected != 'none'
        LEFT JOIN churn ch
            ON c.customer_id = ch.customer_id
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
GROUP BY
    c.customer_id, ch.churned;

churned customer

In [0]:
CREATE OR REPLACE VIEW churned_customer_details AS
SELECT
    c.customer_id,
    c.speed_tier_mbps,
    c.monthly_bill,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats
FROM
    customer c
        LEFT JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected != 'none'
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
        JOIN churn ch
            ON c.customer_id = ch.customer_id
WHERE
    ch.churned = true
GROUP BY
    c.customer_id,
    c.speed_tier_mbps,
    c.monthly_bill;

location detail

In [0]:
CREATE OR REPLACE VIEW location_detail AS
WITH revenue_per_location AS (
    SELECT
        zone,
        SUM(monthly_bill) AS revenue
    FROM
        customer
    GROUP BY
        zone
),
issues_per_location AS (
    SELECT
        c.zone,
        COUNT(
            CASE
                WHEN l.issue_detected != 'none' THEN 1
            END
        ) AS issue_count
    FROM
        customer c
            LEFT JOIN log l
                ON c.customer_id = l.customer_id
    GROUP BY
        c.zone
)
SELECT
    c.zone,
    COUNT(DISTINCT c.customer_id) AS customer_count,
    ROUND(r.revenue / 1000, 2) AS revenue_in_t,
    i.issue_count AS issue_count,
    COUNT(DISTINCT t.ticket_id) AS ticket_count,
    COUNT(DISTINCT ch.session_id) AS chat_count
FROM
    customer c
        LEFT JOIN revenue_per_location r
            ON c.zone = r.zone
        LEFT JOIN issues_per_location i
            ON c.zone = i.zone
        LEFT JOIN ticket t
            ON c.customer_id = t.customer_id
        LEFT JOIN chat ch
            ON c.customer_id = ch.customer_id
GROUP BY
    c.zone,
    r.revenue,
    i.issue_count;

average connection quality

In [0]:
CREATE OR REPLACE VIEW average_connection_quality AS
SELECT
    c.zone,
    ROUND(AVG(l.speed_measured_mbps), 0) AS avg_speed,
    ROUND(AVG(l.packet_loss_percent), 2) AS avg_packet_loss,
    ROUND(AVG(l.latency_ms), 2) AS avg_latency,
    ROUND(AVG(l.downtime_minutes), 2) AS avg_downtime,
    ROUND(AVG(l.connection_drops_count), 2) AS avg_connection_drops,
    COUNT(
        CASE
            WHEN l.issue_detected != 'none' THEN 1
            ELSE 0
        END
    ) AS count_issues
FROM
    log l
        LEFT JOIN customer c
            ON l.customer_id = c.customer_id
GROUP BY
    zone;

chat issues

In [0]:
CREATE OR REPLACE VIEW chat_issues AS
SELECT
    c.classification,
    m.sentiment,
    COUNT(m.sentiment) AS count,
    FIRST(c.comment) AS exmp_comment
FROM
    chat c
    LEFT JOIN message m
WHERE
    classification IS NOT NULL
    AND m.speaker = 'customer'
GROUP BY
    c.classification,
    m.sentiment
ORDER BY
    count DESC

sentiment for agent

In [0]:
CREATE OR REPLACE VIEW sentiment_for_agent AS
SELECT
    CONCAT(a.first_name, ' ', a.last_name) AS agent_name,
    m.sentiment,
    COUNT(m.sentiment) AS count
FROM
    chat c
        JOIN message m
            ON c.session_id = m.session_id
            AND m.speaker = 'customer'
        LEFT JOIN agent a
            ON c.agent_id = a.agent_id
GROUP BY
    agent_name,
    m.sentiment;

# Show tables

In [0]:
SELECT * FROM average_customer;

zone,amount_customers,avg_monthly_bill,avg_speed_tier,avg_data_usage
Thüringen,5672,75.71,305.0,298.4
Schleswig-Holstein,9586,75.75,307.0,296.62
Hamburg,3353,76.08,310.0,291.14
Sachsen,12536,75.66,304.0,298.26
Hessen,42,72.7,285.0,311.85
Berlin,4793,76.14,313.0,298.88
null,1609,76.33,316.0,297.57
Nordrhein-Westfalen,52054,75.92,306.0,302.02
Brandenburg,10132,75.72,305.0,299.58
Niedersachsen,1,119.99,1000.0,838.1


In [0]:
SELECT * FROM customer_connection_ticket_count ORDER BY tickets DESC LIMIT 20;

customer_id,connection_fails,tickets,chats,churned
CUST_01086,4,3,0,false
CUST_00739,13,3,2,false
CUST_00971,9,3,0,true
CUST_00865,6,3,2,false
CUST_01021,5,3,0,false
CUST_01051,5,3,0,false
CUST_00274,7,2,1,false
CUST_00003,8,2,0,true
CUST_00803,6,2,0,false
CUST_00761,3,2,0,false


In [0]:
SELECT * FROM churned_customer_details ORDER BY tickets DESC LIMIT 20;

customer_id,speed_tier_mbps,monthly_bill,connection_fails,tickets,chats
CUST_00971,200,79.99,9,3,0
CUST_01003,1000,119.99,6,2,0
CUST_00284,50,49.99,4,2,0
CUST_00021,200,71.991,5,2,0
CUST_00003,200,71.991,8,2,0
CUST_00784,50,49.99,5,2,2
CUST_00695,50,49.99,4,2,0
CUST_00106,50,49.99,10,2,1
CUST_00577,50,44.991,2,2,2
CUST_00615,1000,119.99,14,2,0


In [0]:
SELECT * FROM location_detail ORDER BY customer_count DESC;

zone,customer_count,revenue_in_t,issue_count,ticket_count,chat_count
Nordrhein-Westfalen,52054,3951.91,31639,242,251
Sachsen,12536,948.46,7695,51,57
Brandenburg,10132,767.17,5791,58,58
Schleswig-Holstein,9586,726.19,6135,53,49
Thüringen,5672,429.41,3521,36,25
Berlin,4793,364.95,2872,29,19
Hamburg,3353,255.11,2233,20,20
null,1609,null,null,9,14
Bayern,102,7.74,47,0,2
Sachsen-Anhalt,80,6.01,25,0,0


In [0]:
SELECT * FROM average_connection_quality;

zone,avg_speed,avg_packet_loss,avg_latency,avg_downtime,avg_connection_drops,count_issues
Thüringen,246.0,3.5,47.83,3.31,2.72,19165
Saarland,45.0,1.57,38.52,0.0,0.08,26
Bayern,114.0,3.35,51.15,2.28,2.42,223
Schleswig-Holstein,272.0,3.37,47.29,3.1,2.51,34101
Sachsen,249.0,3.42,47.27,3.21,2.65,42845
Hamburg,250.0,3.29,47.76,2.92,2.33,12240
Hessen,157.0,2.71,42.6,3.27,2.53,220
Berlin,252.0,3.29,46.8,2.99,2.38,16359
null,309.0,3.31,46.5,3.04,2.53,6000
Mecklenburg-Vorpommern,167.0,6.34,31.96,11.73,4.0,44


In [0]:
SELECT * FROM chat_issues order by classification, sentiment;

classification,sentiment,count,exmp_comment


In [0]:
SELECT * FROM sentiment_for_agent ORDER BY agent_name, sentiment LIMIT 20;

agent_name,sentiment,count
